# Splice — Adaptive Movie Recommender

A movie recommender that switches strategy based on how much is known about a user:

| Ratings known | Engine |
|---|---|
| 0 | Popularity ranking (gender-filtered) |
| 1–4 | Item-based KNN (cosine similarity) |
| 5+ | Stacked ensemble (5 CF models → LightGBM meta-learner) |

This notebook loads the already-trained models from `models/` and walks through
how the system works. All the actual logic lives in `src/` — this notebook is
a guided tour, not where the modelling happens.

If you want to retrain from scratch, see the last section.

## Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src.*` works when running from notebooks/

import pickle
import pandas as pd

from src.data import load_and_clean
from src.engines import (
    get_popularity_recs,
    get_knn_recs,
    build_ensemble_predict,
    recommend,
)
from src.evaluate import evaluate_on_test, precision_recall_at_k, catalog_coverage

## Load data and trained models

`load_and_clean()` re-reads and re-merges the raw TSVs — this part is cheap,
so we just rerun it rather than pickling the master tables. The actual trained
models (which took several minutes of CV to produce) are loaded from disk.

In [ ]:
data = load_and_clean()
train_master = data["train_master"]
test_master = data["test_master"]
movie_info = data["movie_info"]
user_info = data["user_info"]

print(f"Train: {train_master.shape}, Test: {test_master.shape}")

In [ ]:
with open("../models/base_models.pkl", "rb") as f:
    base_models = pickle.load(f)

with open("../models/meta_learner.pkl", "rb") as f:
    meta = pickle.load(f)

with open("../models/popularity_lists.pkl", "rb") as f:
    popularity_lists = pickle.load(f)

with open("../models/knn.pkl", "rb") as f:
    knn = pickle.load(f)

with open("../models/stat_features.pkl", "rb") as f:
    stat_features = pickle.load(f)

print("Base models:", list(base_models.keys()))

In [ ]:
ensemble_predict = build_ensemble_predict(
    base_models,
    meta,
    stat_features["user_stats"],
    stat_features["item_stats"],
    stat_features["global_mean"],
)

## A note on SVD++

The original model set included SVD++, a variant of SVD that also factors in
implicit feedback (which items a user rated, regardless of the rating value).
It's usually one of the stronger models on MovieLens-100k, so it was included
as a base learner going in.

On out-of-fold evaluation it came back with an RMSE of **1.0755** — worse than
the plain `Baseline` model (0.9464), and the weakest of everything tested. Rather
than let it drag the ensemble down (or silently get near-zero weight from the
meta-learner while still costing the most training time of any model), it was
dropped. The final base model set is SVD, item-based KNN, Baseline, SlopeOne,
and Co-Clustering.

Worth trying differently-tuned SVD++ hyperparameters at some point (see
`GitHub Issues`), but for now the simpler five-model ensemble outperforms it.

## Base model performance (out-of-fold)

In [ ]:
with open("../models/results.txt") as f:
    print(f.read())

## Cold start — the popularity engine

For a brand-new user with zero ratings, there's no history to work from, so
we fall back to simple popularity, optionally filtered by gender.

In [ ]:
print("Top 10 overall:")
for title in get_popularity_recs(popularity_lists, gender=None, n=10):
    print(" -", title)

## Sparse users — item-based KNN

Once a user has a few ratings (1–4), there's enough signal to do item-item
similarity off their highest-rated movie, without needing the full ensemble.

In [ ]:
example_movie_id = 1  # Toy Story
print(f"Because you liked movie {example_movie_id}:")
for title in get_knn_recs(example_movie_id, knn, movie_info, n=10):
    print(" -", title)

## Established users — the full ensemble

For users with 5+ ratings, every candidate movie gets scored by all five base
models, and the meta-learner combines those scores (plus user/item bias
features) into a final prediction.

In [ ]:
example_user_id = 196
recs = recommend(
    example_user_id, train_master, movie_info, popularity_lists, knn,
    ensemble_predict, user_info=user_info, n=10,
)

for movie_id, title, score in recs:
    print(f"  {score:.2f}  {title}")

## Evaluation

`evaluate_on_test` runs the held-out 20,000-rating test set through the
ensemble exactly once — this is the only place `test_master` gets used for
anything other than loading.

In [ ]:
test_rmse = evaluate_on_test(test_master, train_master, ensemble_predict)

### Ranking quality — precision/recall @10

RMSE tells you how close individual rating predictions are, but the actual
product is a ranked list. Precision@10 asks: of the top 10 recommended movies,
how many did the user actually rate 4 or 5 stars? Recall@10 asks: of all the
movies this user rated highly, how many showed up in the top 10?

In [ ]:
pr = precision_recall_at_k(test_master, train_master, ensemble_predict, k=10)
print(f"Precision@10: {pr['precision_at_k']:.3f}")
print(f"Recall@10:    {pr['recall_at_k']:.3f}")
print(f"Evaluated across {pr['users_evaluated']} test users")

## Retraining from scratch

Not needed to explore this notebook — the cells above load pre-trained models.
Only run this if you've changed something in `src/` and want fresh results.
This takes several minutes (10-fold CV across 5 models).

In [ ]:
# from src.train import main
# main()